# Find matches

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import geopandas as gpd
import pandas as pd
import numpy as np
import os
import rasterio as rio 

from datetime import datetime, timedelta
from pyproj import CRS
from rasterio.warp import transform_bounds
from shapely.geometry import Polygon
from tqdm import tqdm

In [3]:
from sensorsio.mgrs import get_bbox_mgrs_tile, get_mgrs_tiles_from_roi

In [4]:
import sys
sys.path.append("../src")

In [5]:
from etdataset.database import create_ls8_db, create_eco_db, select_products, to_collectionV2

### Inputs

In [6]:
if not os.environ.get("DATA_PATH"):
    raise Exception("DATA_PATH not defined")

In [7]:
data_path = os.environ["DATA_PATH"]

In [8]:
# Databases paths
ls8_db_path = os.path.join(data_path,"databases/landsat_ot_c2_l2_659ed86540bcec65.csv")
eco_db_path = os.path.join(data_path,"databases/ecostress_eco2lste_655f877cf0476b91.csv")
s2_db_path = os.path.join(data_path,"databases/SENTINEL2.csv")

In [9]:
# ROI
s2_tile_id = "28PCA"
roi_path = os.path.join(data_path,"zones/Zone_Senegal_Centre.shp") 

In [10]:
# Acquisition dates
min_date_str = "2023-02-01" # Format YYYY-MM-DD
max_date_str = "2023-03-31" # Format YYYY-MM-DD
delta_time_str = "3 days" # Delta time between two acquisitions

In [11]:
cloud_cover = 25
tile_coverage = 33

In [12]:
output_path = "out"
os.makedirs(output_path,exist_ok=True)

In [13]:
# Convert dates
min_date = datetime.strptime(min_date_str,"%Y-%m-%d")
max_date = datetime.strptime(max_date_str,"%Y-%m-%d")
delta = pd.Timedelta(delta_time_str)

In [14]:
s2_tile_bbox = get_bbox_mgrs_tile(s2_tile_id)
s2_tile_crs = 4326

In [15]:
roi = gpd.read_file(roi_path)
roi_crs = roi.crs
roi_bbox = rio.coords.BoundingBox(*roi.bounds.iloc[0].values) 
wgs84_bounds = transform_bounds(roi_crs, 4326, *roi_bbox)
roi_poly = Polygon([[wgs84_bounds[0], wgs84_bounds[1]], [wgs84_bounds[0], wgs84_bounds[3]],
                    [wgs84_bounds[2], wgs84_bounds[3]], [wgs84_bounds[2], wgs84_bounds[1]]])

In [16]:
mgrs_tiles = get_mgrs_tiles_from_roi(roi_bbox,roi_crs)

In [17]:
mgrs_tiles

,Name,overlap_geometry,geometry,overlap_percentage
0,28PBA,"POLYGON ((-16.76482 14.46621, -16.75975 13.800...","POLYGON ((-17.78282 14.45638, -16.76482 14.466...",22.271238
1,28PBB,"POLYGON ((-16.76412 14.37783, -17.10000 14.374...","POLYGON ((-17.79446 15.35961, -16.77221 15.370...",20.551890
2,28PCA,"POLYGON ((-15.90000 14.47098, -15.90000 13.800...","POLYGON ((-16.85552 14.46551, -15.83696 14.471...",63.120620
3,28PCB,"POLYGON ((-16.85479 14.37713, -16.86012 15.000...","POLYGON ((-16.86329 15.36934, -15.84047 15.375...",58.627789
4,28PDA,"POLYGON ((-15.90000 14.47105, -15.90000 13.800...","POLYGON ((-15.92826 14.47100, -14.90943 14.472...",1.789794
5,28PDB,"POLYGON ((-15.92790 14.38258, -15.93054 15.000...","POLYGON ((-15.93215 15.37519, -14.90905 15.377...",1.780280


### Find Landsat products

In [18]:
%%time
ls8_db = create_ls8_db(ls8_db_path, 
                       min_date = min_date, 
                       max_date = max_date, 
                       max_cloud_cover=cloud_cover,                    
                       roi_bbox = roi_bbox,
                       roi_crs = roi_crs,
                       min_roi_overlap=80)

24-01-16 22:54:39 :: INFO :: Found 6 Landsat products fullfilling the requirements


CPU times: user 27 s, sys: 2.02 s, total: 29 s
Wall time: 29.1 s


In [19]:
ls8_db

,product_name,date,cloud_cover,overlap_geometry,geometry,overlap_percentage
0,LC09_L2SP_205050_20230207_20230311_02_T1,2023-02-07,19.05,"POLYGON ((-17.10000 15.00000, -15.90000 15.000...","POLYGON ((-17.78678 15.49572, -15.65362 15.512...",100.0
1,LC09_L2SP_205050_20230223_20230308_02_T1,2023-02-23,0.31,"POLYGON ((-17.10000 15.00000, -15.90000 15.000...","POLYGON ((-17.74767 15.49621, -15.61446 15.512...",100.0
2,LC08_L2SP_205050_20230303_20230316_02_T1,2023-03-03,2.21,"POLYGON ((-17.10000 15.00000, -15.90000 15.000...","POLYGON ((-17.73373 15.49909, -15.60328 15.515...",100.0
3,LC09_L2SP_205050_20230311_20230313_02_T1,2023-03-11,0.04,"POLYGON ((-17.10000 15.00000, -15.90000 15.000...","POLYGON ((-17.73929 15.49632, -15.60607 15.512...",100.0
4,LC08_L2SP_205050_20230319_20230324_02_T1,2023-03-19,0.66,"POLYGON ((-17.10000 15.00000, -15.90000 15.000...","POLYGON ((-17.73932 15.49902, -15.60607 15.515...",100.0
5,LC09_L2SP_205050_20230327_20230329_02_T1,2023-03-27,0.07,"POLYGON ((-17.10000 15.00000, -15.90000 15.000...","POLYGON ((-17.73370 15.49638, -15.60047 15.512...",100.0


### Find ECOSTRESS products

In [20]:
%%time
eco_db = create_eco_db(eco_db_path, 
                       min_date = min_date, 
                       max_date = max_date, 
                       roi_bbox = roi_bbox,
                       roi_crs = roi_crs,
                       min_roi_overlap=90)

24-01-16 22:54:43 :: INFO :: Found 14 ECOSTRESS products fullfilling the requirements


CPU times: user 3.36 s, sys: 152 ms, total: 3.51 s
Wall time: 3.52 s


In [21]:
eco_db

,product_name,date,overlap_geometry,geometry,overlap_percentage
0,ECOSTRESS_L2_LSTE_26372_016_20230301T111607_06...,2023-03-01,"POLYGON ((-17.10000 15.00000, -15.90000 15.000...","POLYGON ((-19.54500 16.40010, -14.69010 16.400...",100.000000
1,ECOSTRESS_L2_LSTE_26381_003_20230302T003454_06...,2023-03-02,"POLYGON ((-17.10000 15.00000, -15.90000 15.000...","POLYGON ((-19.82070 17.05810, -14.66490 17.058...",100.000000
2,ECOSTRESS_L2_LSTE_26433_014_20230305T094027_06...,2023-03-05,"POLYGON ((-17.10000 15.00000, -15.90000 15.000...","POLYGON ((-20.16460 17.74720, -15.28730 17.747...",100.000000
3,ECOSTRESS_L2_LSTE_26433_015_20230305T094119_06...,2023-03-05,"POLYGON ((-17.10000 15.00000, -15.90000 15.000...","POLYGON ((-18.14920 15.15320, -13.32340 15.153...",100.000000
4,ECOSTRESS_L2_LSTE_26494_014_20230309T080538_06...,2023-03-09,"POLYGON ((-17.10000 15.00000, -15.90000 15.000...","POLYGON ((-20.50760 18.53890, -15.59570 18.538...",100.000000
5,ECOSTRESS_L2_LSTE_26494_015_20230309T080630_06...,2023-03-09,"POLYGON ((-17.10000 15.00000, -15.90000 15.000...","POLYGON ((-18.47280 15.94490, -13.61960 15.944...",100.000000
6,ECOSTRESS_L2_LSTE_26616_002_20230317T045856_06...,2023-03-17,"POLYGON ((-17.10000 15.00000, -15.90000 15.000...","POLYGON ((-18.93940 16.74560, -14.09750 16.745...",100.000000
7,ECOSTRESS_L2_LSTE_26678_018_20230321T032452_06...,2023-03-21,"POLYGON ((-17.10000 15.00000, -15.90000 15.000...","POLYGON ((-19.00990 17.01900, -14.16570 17.019...",100.000000
8,ECOSTRESS_L2_LSTE_26687_003_20230321T164353_06...,2023-03-21,"POLYGON ((-17.10000 15.00000, -15.90000 15.000...","POLYGON ((-19.07110 16.80060, -13.89560 16.800...",100.000000
9,ECOSTRESS_L2_LSTE_26739_017_20230325T015012_06...,2023-03-25,"POLYGON ((-17.10000 15.00000, -15.90000 15.000...","POLYGON ((-19.11100 17.51650, -14.25820 17.516...",100.000000


In [22]:
eco_db_v2 = to_collectionV2(eco_db,roi_bbox, roi_crs,30)

In [23]:
eco_db_v2

,product_name,date,overlap_geometry,overlap_percentage,geometry
0,ECOv002_L2T_LSTE_26372_016_28PCA_20230301T1116...,2023-03-01,"POLYGON ((-15.9 14.470980835362138, -15.9 13.8...",63.120620,"POLYGON ((-16.85552 14.46551, -15.83696 14.471..."
1,ECOv002_L2T_LSTE_26372_016_28PCB_20230301T1116...,2023-03-01,"POLYGON ((-16.8547891789 14.3771328743, -16.86...",58.627789,"POLYGON ((-16.86329 15.36934, -15.84047 15.375..."
2,ECOv002_L2T_LSTE_26381_003_28PCA_20230302T0034...,2023-03-02,"POLYGON ((-15.9 14.470980835362138, -15.9 13.8...",63.120620,"POLYGON ((-16.85552 14.46551, -15.83696 14.471..."
3,ECOv002_L2T_LSTE_26381_003_28PCB_20230302T0034...,2023-03-02,"POLYGON ((-16.8547891789 14.3771328743, -16.86...",58.627789,"POLYGON ((-16.86329 15.36934, -15.84047 15.375..."
4,ECOv002_L2T_LSTE_26433_014_28PCA_20230305T0940...,2023-03-05,"POLYGON ((-15.9 14.470980835362138, -15.9 13.8...",63.120620,"POLYGON ((-16.85552 14.46551, -15.83696 14.471..."
5,ECOv002_L2T_LSTE_26433_014_28PCB_20230305T0940...,2023-03-05,"POLYGON ((-16.8547891789 14.3771328743, -16.86...",58.627789,"POLYGON ((-16.86329 15.36934, -15.84047 15.375..."
6,ECOv002_L2T_LSTE_26433_015_28PCA_20230305T0941...,2023-03-05,"POLYGON ((-15.9 14.470980835362138, -15.9 13.8...",63.120620,"POLYGON ((-16.85552 14.46551, -15.83696 14.471..."
7,ECOv002_L2T_LSTE_26433_015_28PCB_20230305T0941...,2023-03-05,"POLYGON ((-16.8547891789 14.3771328743, -16.86...",58.627789,"POLYGON ((-16.86329 15.36934, -15.84047 15.375..."
8,ECOv002_L2T_LSTE_26494_014_28PCA_20230309T0805...,2023-03-09,"POLYGON ((-15.9 14.470980835362138, -15.9 13.8...",63.120620,"POLYGON ((-16.85552 14.46551, -15.83696 14.471..."
9,ECOv002_L2T_LSTE_26494_014_28PCB_20230309T0805...,2023-03-09,"POLYGON ((-16.8547891789 14.3771328743, -16.86...",58.627789,"POLYGON ((-16.86329 15.36934, -15.84047 15.375..."


### Select matches 

In [24]:
res = select_products(ls8_db,eco_db_v2,delta, 40)

Matching datasets ...: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6/6 [00:00<00:00, 88.94it/s]
24-01-16 22:54:43 :: INFO :: Found 28 matches


In [25]:
res

,product_name_1,date_1,cloud_cover,overlap_geometry_1,overlap_percentage_1,product_name_2,date_2,overlap_geometry_2,overlap_percentage_2,geometry,overlap
0,LC08_L2SP_205050_20230303_20230316_02_T1,2023-03-03,2.21,"POLYGON ((-17.10000 15.00000, -15.90000 15.000...",100.0,ECOv002_L2T_LSTE_26372_016_28PCA_20230301T1116...,2023-03-01,"POLYGON ((-15.9 14.470980835362138, -15.9 13.8...",63.120620,"POLYGON ((-15.83696 14.47134, -15.83339 13.478...",100.0
1,LC08_L2SP_205050_20230303_20230316_02_T1,2023-03-03,2.21,"POLYGON ((-17.10000 15.00000, -15.90000 15.000...",100.0,ECOv002_L2T_LSTE_26372_016_28PCB_20230301T1116...,2023-03-01,"POLYGON ((-16.8547891789 14.3771328743, -16.86...",58.627789,"POLYGON ((-15.84047 15.37555, -15.83663 14.382...",100.0
2,LC08_L2SP_205050_20230303_20230316_02_T1,2023-03-03,2.21,"POLYGON ((-17.10000 15.00000, -15.90000 15.000...",100.0,ECOv002_L2T_LSTE_26381_003_28PCA_20230302T0034...,2023-03-02,"POLYGON ((-15.9 14.470980835362138, -15.9 13.8...",63.120620,"POLYGON ((-15.83696 14.47134, -15.83339 13.478...",100.0
3,LC08_L2SP_205050_20230303_20230316_02_T1,2023-03-03,2.21,"POLYGON ((-17.10000 15.00000, -15.90000 15.000...",100.0,ECOv002_L2T_LSTE_26381_003_28PCB_20230302T0034...,2023-03-02,"POLYGON ((-16.8547891789 14.3771328743, -16.86...",58.627789,"POLYGON ((-15.84047 15.37555, -15.83663 14.382...",100.0
4,LC08_L2SP_205050_20230303_20230316_02_T1,2023-03-03,2.21,"POLYGON ((-17.10000 15.00000, -15.90000 15.000...",100.0,ECOv002_L2T_LSTE_26433_014_28PCA_20230305T0940...,2023-03-05,"POLYGON ((-15.9 14.470980835362138, -15.9 13.8...",63.120620,"POLYGON ((-15.83696 14.47134, -15.83339 13.478...",100.0
5,LC08_L2SP_205050_20230303_20230316_02_T1,2023-03-03,2.21,"POLYGON ((-17.10000 15.00000, -15.90000 15.000...",100.0,ECOv002_L2T_LSTE_26433_014_28PCB_20230305T0940...,2023-03-05,"POLYGON ((-16.8547891789 14.3771328743, -16.86...",58.627789,"POLYGON ((-15.84047 15.37555, -15.83663 14.382...",100.0
6,LC08_L2SP_205050_20230303_20230316_02_T1,2023-03-03,2.21,"POLYGON ((-17.10000 15.00000, -15.90000 15.000...",100.0,ECOv002_L2T_LSTE_26433_015_28PCA_20230305T0941...,2023-03-05,"POLYGON ((-15.9 14.470980835362138, -15.9 13.8...",63.120620,"POLYGON ((-15.83696 14.47134, -15.83339 13.478...",100.0
7,LC08_L2SP_205050_20230303_20230316_02_T1,2023-03-03,2.21,"POLYGON ((-17.10000 15.00000, -15.90000 15.000...",100.0,ECOv002_L2T_LSTE_26433_015_28PCB_20230305T0941...,2023-03-05,"POLYGON ((-16.8547891789 14.3771328743, -16.86...",58.627789,"POLYGON ((-15.84047 15.37555, -15.83663 14.382...",100.0
8,LC09_L2SP_205050_20230311_20230313_02_T1,2023-03-11,0.04,"POLYGON ((-17.10000 15.00000, -15.90000 15.000...",100.0,ECOv002_L2T_LSTE_26494_014_28PCA_20230309T0805...,2023-03-09,"POLYGON ((-15.9 14.470980835362138, -15.9 13.8...",63.120620,"POLYGON ((-15.83696 14.47134, -15.83339 13.478...",100.0
9,LC09_L2SP_205050_20230311_20230313_02_T1,2023-03-11,0.04,"POLYGON ((-17.10000 15.00000, -15.90000 15.000...",100.0,ECOv002_L2T_LSTE_26494_014_28PCB_20230309T0805...,2023-03-09,"POLYGON ((-16.8547891789 14.3771328743, -16.86...",58.627789,"POLYGON ((-15.84047 15.37555, -15.83663 14.382...",100.0


In [26]:
res['product_name_1'].unique()

array(['LC08_L2SP_205050_20230303_20230316_02_T1',
       'LC09_L2SP_205050_20230311_20230313_02_T1',
       'LC08_L2SP_205050_20230319_20230324_02_T1',
       'LC09_L2SP_205050_20230327_20230329_02_T1'], dtype=object)

In [27]:
res['product_name_2'].unique()

array(['ECOv002_L2T_LSTE_26372_016_28PCA_20230301T111607_0601_02',
       'ECOv002_L2T_LSTE_26372_016_28PCB_20230301T111607_0601_02',
       'ECOv002_L2T_LSTE_26381_003_28PCA_20230302T003454_0601_02',
       'ECOv002_L2T_LSTE_26381_003_28PCB_20230302T003454_0601_02',
       'ECOv002_L2T_LSTE_26433_014_28PCA_20230305T094027_0601_01',
       'ECOv002_L2T_LSTE_26433_014_28PCB_20230305T094027_0601_01',
       'ECOv002_L2T_LSTE_26433_015_28PCA_20230305T094119_0601_01',
       'ECOv002_L2T_LSTE_26433_015_28PCB_20230305T094119_0601_01',
       'ECOv002_L2T_LSTE_26494_014_28PCA_20230309T080538_0601_01',
       'ECOv002_L2T_LSTE_26494_014_28PCB_20230309T080538_0601_01',
       'ECOv002_L2T_LSTE_26494_015_28PCA_20230309T080630_0601_01',
       'ECOv002_L2T_LSTE_26494_015_28PCB_20230309T080630_0601_01',
       'ECOv002_L2T_LSTE_26616_002_28PCA_20230317T045856_0601_02',
       'ECOv002_L2T_LSTE_26616_002_28PCB_20230317T045856_0601_02',
       'ECOv002_L2T_LSTE_26678_018_28PCA_20230321T032452_0601_

In [28]:
for product_name_1, group in res.groupby("product_name_1"):
    print(product_name_1)
    print(group["product_name_2"].unique())

LC08_L2SP_205050_20230303_20230316_02_T1
['ECOv002_L2T_LSTE_26372_016_28PCA_20230301T111607_0601_02'
 'ECOv002_L2T_LSTE_26372_016_28PCB_20230301T111607_0601_02'
 'ECOv002_L2T_LSTE_26381_003_28PCA_20230302T003454_0601_02'
 'ECOv002_L2T_LSTE_26381_003_28PCB_20230302T003454_0601_02'
 'ECOv002_L2T_LSTE_26433_014_28PCA_20230305T094027_0601_01'
 'ECOv002_L2T_LSTE_26433_014_28PCB_20230305T094027_0601_01'
 'ECOv002_L2T_LSTE_26433_015_28PCA_20230305T094119_0601_01'
 'ECOv002_L2T_LSTE_26433_015_28PCB_20230305T094119_0601_01']
LC08_L2SP_205050_20230319_20230324_02_T1
['ECOv002_L2T_LSTE_26616_002_28PCA_20230317T045856_0601_02'
 'ECOv002_L2T_LSTE_26616_002_28PCB_20230317T045856_0601_02'
 'ECOv002_L2T_LSTE_26678_018_28PCA_20230321T032452_0601_02'
 'ECOv002_L2T_LSTE_26678_018_28PCB_20230321T032452_0601_02'
 'ECOv002_L2T_LSTE_26687_003_28PCA_20230321T164353_0601_01'
 'ECOv002_L2T_LSTE_26687_003_28PCB_20230321T164353_0601_01']
LC09_L2SP_205050_20230311_20230313_02_T1
['ECOv002_L2T_LSTE_26494_014_28PCA_

### Write results

In [29]:
res.to_csv(os.path.join(output_path,"ecols8_matches.csv"))